# Image Control Techniques

**Module:** 17 — Image Generation

ControlNet, LoRA, DreamBooth, fine-tuning, and style transfer — precise steering beyond text.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain ControlNet-style spatial conditioning
- Contrast LoRA vs DreamBooth vs full fine-tunes
- Pick a control recipe for pose, layout, product, or identity
- Estimate adapter parameter/cost trade-offs


## ControlNet

### Definition
**ControlNet** attaches a parallel conditioning branch (edges, depth, pose, etc.) so spatial structure is preserved while appearance follows the prompt.

### Why it matters
Text alone is bad at exact layout.

### How it works
Extract control map → conditioned branch → generate with shared prompt/seed.

### Intuition
A lightbox tracing plate under drawing paper.

### Pitfalls
- Over-strong weights stamp lifeless copies
- Mismatched control type
- Resolution misaligned to latent grid

### When to use
Pose transfer, UI layout, architecture, product silhouettes.


### Control map menu

| Map | Locks | Good for |
|-----|-------|----------|
| Canny / lineart | Edges | Outlines |
| Depth | 3D layout | Rooms, landscapes |
| OpenPose | Joints | Character posing |
| Scribble | Loose structure | Sketch ideation |
| Segmentation | Region classes | Layout-aware scenes |

```mermaid
flowchart LR
  G[Guide] --> M[Control map]
  M --> CN[ControlNet]
  P[Prompt] --> BASE[Denoiser]
  CN --> BASE --> OUT[Image]
```


In [ ]:
# Demo 1: control weight regimes + pose payload
def regime(control_weight: float) -> str:
    if control_weight >= 1.2: return "structure_dominated"
    if control_weight <= 0.4: return "prompt_dominated"
    return "balanced"

for w in [0.3, 0.8, 1.4]:
    print(w, regime(w))
pose_json = {"people": [{"pose_keypoints_2d": [100, 200, 0.9, 120, 240, 0.8]}], "canvas": {"width": 512, "height": 768}}
print("keypoints", len(pose_json["people"][0]["pose_keypoints_2d"]))


## LoRA

### Definition
**LoRA** fine-tunes low-rank matrices so a small file steers style/character/product without full checkpoints.

### Why it matters
Cheap multi-tenant customization.

### How it works
Train on curated set; export rank-r; scale α at inference; stack carefully.

### Intuition
A clip-on lens — changes look without replacing the camera body.

### Pitfalls
- Stacking wars → artifacts
- Duplicate near-copies → memorization
- Wrong base version

### When to use
Styles, products, characters with many variants.


In [ ]:
# Demo 2: LoRA parameter estimate
def lora_params(n_matrices: int, dim: int, rank: int) -> int:
    return n_matrices * (dim * rank + rank * dim)

print("rank 8", lora_params(100, 1024, 8))
print("rank 32", lora_params(100, 1024, 32))
print("full matrices", 100 * 1024 * 1024)


## DreamBooth, Fine-Tuning, Style Recipes

### Definition
**DreamBooth** personalizes with few images + unique token (+ prior preservation). **Full FT** for heavy domain shift. Style via NST/adapters/img2img/IP-Adapter.

### Why it matters
When LoRA is not enough or you need a specialized checkpoint.

### How it works
Curate → train instance prompt → regularize class prior → evaluate identity vs promptability.

### Intuition
Teaching an artist your mascot without forgetting 'dog' in general.

### Pitfalls
- Overfit backgrounds
- Token collision
- High img2img strength burns identity
- Copyrighted style refs against policy

### When to use
Subject identity and campaign variants with rights cleared.


In [ ]:
# Demo 3: dataset card + prior preservation bookkeeping
from collections import Counter

def dataset_report(paths_labels):
    c = Counter(lbl for _, lbl in paths_labels)
    return {"n": len(paths_labels), "by_label": dict(c),
            "warn_bg_collapse": c.most_common(1)[0][1] / len(paths_labels) > 0.6}

data = [(f"img{i}.png", "instance") for i in range(8)] + [(f"reg{i}.png", "class_prior") for i in range(20)]
print(dataset_report(data))


### Practical recipes

| Goal | Recipe |
|------|--------|
| Exact pose, new costume | OpenPose + character LoRA |
| Product new background | Seg/inpaint + product lock |
| Brand style across SKUs | Style LoRA + brand block |
| Face likeness (rights OK) | Face adapter / DreamBooth + holdout |
| Sketch → render | Scribble/lineart ControlNet |

Fine-tune ladder: `prompt → LoRA → DreamBooth/adapters → full FT`


In [ ]:
# Demo 4: recipe router
def recommend(goal: str) -> str:
    g = goal.lower()
    if "pose" in g: return "controlnet_openpose + optional character lora"
    if "product" in g and "bg" in g: return "seg/inpaint + product locked"
    if "style" in g: return "style lora + brand block; strength sweep"
    if "identity" in g: return "dreambooth/ip-adapter + identity holdout"
    return "start with prompt+seed sweep"

for g in ["lock pose for dance", "product new bg", "brand style set", "identity mascot"]:
    print(g, "=>", recommend(g))


### Try it yourself — Control

1. Design control JSON: map_type, weight, start_step, end_step.
2. Estimate LoRA sizes for ranks {4,8,16,32}.
3. Write an eval sheet: identity, promptability, artifacts.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `ControlNet` | Spatial conditioning branch for diffusion |
| `LoRA` | Low-rank adapter fine-tune |
| `DreamBooth` | Few-shot subject personalization |
| `IP-Adapter` | Reference-image conditioning adapter |
| `prior preservation` | Regularization keeping class diversity |


### Workshop — Parameter journal — Image Control

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Control
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Control

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Control
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Control

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Control
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Control

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Control
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Control

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Control
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Control

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Control
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Image Control

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Image Control
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Image Control

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Image Control
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Spatial controls fix what text cannot
- LoRA is the default customization unit
- Recipes beat isolated tools — combine lightly
